In [1]:
import numpy as np
import pandas as pd
import os
import re
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate  
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    recall_score,
    roc_auc_score,
    precision_score,
    f1_score
)

In [45]:
#Revision 3 testing: save the train and test to seperate files to ensure no data leakage
# Define directories
# need to set up output_dir and target_variable
input_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals v2\post baseline correction"
output_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results_final"
split_data_dir = os.path.join(output_dir, "train_test_splits")
os.makedirs(split_data_dir, exist_ok=True) 
random_state = 12

# Create output directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(split_data_dir, exist_ok=True)

# Variables to predict
target_variable = "q1_y"
class_names = {0: "neutral", 1: "emotional"}  # Mapping for readable output
excluded_columns = ["q1_y", "q2_y", "q3_y", "q4_y", "stimID"]
special_columns = ["Neutral", "Happy", "Sad", "Angry", "Surprised", "Scared", "Disgusted", "Valence", "Arousal"]

# Initialize comprehensive summary
summary_data = []
grand_total = {
    'original': {0: 0, 1: 0},
    'train': {0: 0, 1: 0},
    'test': {0: 0, 1: 0}
}

print("=== Creating train/test splits and tracking sample sizes ===")
for file_name in os.listdir(input_dir):
    if file_name.endswith(".csv"):
        file_path = os.path.join(input_dir, file_name)
        participant_id = file_name.split('.')[0]
        
        df = pd.read_csv(file_path, dtype={target_variable: str})
        
        # Data cleaning
        for col in special_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: 0 if isinstance(x, str) and 'F' in x else (x if isinstance(x, (int, float)) else 0))

        y = df[target_variable].replace({
            "long_happy": 4, "long_sad": 3, "fixation": 2,
            "emotional": 1, "neutral": 0, "Emotional": 1, "Neutral": 0
        })
        
        mask = (y == 1) | (y == 0)
        X = df.drop(columns=excluded_columns)[mask]
        y = y[mask].astype(int)
        
        if y.nunique() < 2:
            print(f"Skipping {participant_id} - only one class present")
            continue
        
        # Get counts
        original_counts = y.value_counts().to_dict()
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=random_state, stratify=y)
        train_counts = y_train.value_counts().to_dict()
        test_counts = y_test.value_counts().to_dict()
        
        # Update grand totals
        for class_val in [0, 1]:
            grand_total['original'][class_val] += original_counts.get(class_val, 0)
            grand_total['train'][class_val] += train_counts.get(class_val, 0)
            grand_total['test'][class_val] += test_counts.get(class_val, 0)
        
        # Add to participant summary
        summary_data.append({
            'participant': participant_id,
            'original_neutral': original_counts.get(0, 0),
            'original_emotional': original_counts.get(1, 0),
            'train_neutral': train_counts.get(0, 0),
            'train_emotional': train_counts.get(1, 0),
            'test_neutral': test_counts.get(0, 0),
            'test_emotional': test_counts.get(1, 0),
            'original_ratio': original_counts.get(0, 0)/original_counts.get(1, 0) if original_counts.get(1, 0) != 0 else float('inf'),
            'train_ratio': train_counts.get(0, 0)/train_counts.get(1, 0) if train_counts.get(1, 0) != 0 else float('inf'),
            'test_ratio': test_counts.get(0, 0)/test_counts.get(1, 0) if test_counts.get(1, 0) != 0 else float('inf')
        })
        
        # Save splits to files
        train_data = X_train.copy()
        train_data[target_variable] = y_train
        test_data = X_test.copy()
        test_data[target_variable] = y_test
        
        train_path = os.path.join(split_data_dir, f"{participant_id}_train.csv")
        test_path = os.path.join(split_data_dir, f"{participant_id}_test.csv")
        
        # Verify directory exists (ADD THIS)
        if not os.path.exists(split_data_dir):
            raise Exception(f"Directory not created: {split_data_dir}")
            
        train_data.to_csv(train_path, index=False)
        test_data.to_csv(test_path, index=False)
        print(f"Saved splits for participant {participant_id}")

# Create and save detailed summary
summary_df = pd.DataFrame(summary_data)
summary_path = os.path.join(output_dir, "sample_size_summary.csv")
summary_df.to_csv(summary_path, index=False)

# Create grand totals summary
totals_df = pd.DataFrame({
    'dataset': ['original', 'train', 'test'],
    'neutral': [grand_total['original'][0], grand_total['train'][0], grand_total['test'][0]],
    'emotional': [grand_total['original'][1], grand_total['train'][1], grand_total['test'][1]],
    'ratio': [
        grand_total['original'][0]/grand_total['original'][1] if grand_total['original'][1] != 0 else float('inf'),
        grand_total['train'][0]/grand_total['train'][1] if grand_total['train'][1] != 0 else float('inf'),
        grand_total['test'][0]/grand_total['test'][1] if grand_total['test'][1] != 0 else float('inf')
    ]
})
totals_path = os.path.join(output_dir, "grand_totals_summary.csv")
totals_df.to_csv(totals_path, index=False)

print(f"\n=== Summary ===")
print(f"Processed {len(summary_df)} participants")
print(f"Grand totals:\n{totals_df.to_string(index=False)}")
print(f"\nDetailed participant summaries saved to: {summary_path}")
print(f"Aggregated totals saved to: {totals_path}")

=== Creating train/test splits and tracking sample sizes ===


C:\Users\OLten\AppData\Local\Temp\ipykernel_11412\3487487862.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df[target_variable].replace({


Saved splits for participant corrected_11_processed_final


C:\Users\OLten\AppData\Local\Temp\ipykernel_11412\3487487862.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df[target_variable].replace({


Saved splits for participant corrected_59_processed_final


...

Saved splits for participant corrected_20_processed_final


C:\Users\OLten\AppData\Local\Temp\ipykernel_11412\3487487862.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df[target_variable].replace({


Saved splits for participant corrected_03_processed_final


C:\Users\OLten\AppData\Local\Temp\ipykernel_11412\3487487862.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df[target_variable].replace({


Saved splits for participant corrected_21_processed_final

=== Summary ===
Processed 70 participants
Grand totals:
 dataset  neutral  emotional    ratio
original    18980      18407 1.031129
   train    15175      14722 1.030770
    test     3805       3685 1.032564

Detailed participant summaries saved to: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results_final\sample_size_summary.csv
Aggregated totals saved to: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results_final\grand_totals_summary.csv


In [49]:
#############################################################
# Second pass: Load splits and perform analysis
# Define directories
input_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results_final\train_test_splits"
output_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results_final"
os.makedirs(output_dir, exist_ok=True) 

# Configuration
permute_y = True # Set to False for normal classification
random_state = 42  # For reproducibility
cv_folds = 10     # Number of cross-validation folds

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Variables to predict
target_variable = "q1_y"

# Determine report type
if permute_y:
    report_type = "permutation_reports"
    permutation_label = "permutation"
else:
    report_type = "classification_reports"
    permutation_label = "original"

print("\n=== Analyzing saved splits ===")
cv_metrics = []
test_metrics = []
weights_data = []

for file_name in os.listdir(input_dir):
    if file_name.endswith("_train.csv"):  # Look for train files first
        # Extract participant ID (e.g., "01" from "corrected_01_processed_final_train.csv")
        participant_id = file_name.split('_')[1]
        print(f"\nAnalyzing participant {participant_id}")
        
        # Construct file paths
        base_name = f"corrected_{participant_id}_processed_final"
        train_path = os.path.join(input_dir, f"{base_name}_train.csv")
        test_path = os.path.join(input_dir, f"{base_name}_test.csv")
        
        # Verify files exist
        if not os.path.exists(train_path):
            print(f"Warning: Train file not found - {train_path}")
            continue
        if not os.path.exists(test_path):
            print(f"Warning: Test file not found - {test_path}")
            continue
        
        # Load data - TRAIN data for cross-validation
        train_data = pd.read_csv(train_path)
        X_train = train_data.drop(columns=[target_variable])
        y_train = train_data[target_variable]
        
        # Load TEST data for final evaluation
        test_data = pd.read_csv(test_path)
        X_test = test_data.drop(columns=[target_variable])
        y_test = test_data[target_variable]
        
        if permute_y:
            np.random.seed(random_state)
            y_train = np.random.permutation(y_train.values)
            y_test = np.random.permutation(y_test.values)
        
        # Define pipeline
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='linear', random_state=random_state))
        ])
        
        # Perform 10-fold cross-validation on TRAIN data
        print("Performing 10-fold CV on train data...")
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
        cv_results = cross_validate(
            pipeline, 
            X_train,  # Using train data for CV
            y_train,
            cv=cv,
            scoring=['accuracy', 'balanced_accuracy', 'f1', 'roc_auc'],
            return_train_score=False
        )
        
        # Store CV metrics
        cv_metrics.append({
            'Participant': participant_id,
            'CV_Accuracy_Mean': np.mean(cv_results['test_accuracy']),
            'CV_Accuracy_Std': np.std(cv_results['test_accuracy']),
            'CV_Balanced_Accuracy_Mean': np.mean(cv_results['test_balanced_accuracy']),
            'CV_F1_Mean': np.mean(cv_results['test_f1']),
            'CV_ROC_AUC_Mean': np.mean(cv_results['test_roc_auc'])
        })
        
        # Final training on full TRAIN data and evaluation on TEST data
        print("Training on full train data and evaluating on test data...")
        pipeline.fit(X_train, y_train)
        
        # Get feature weights
        svm = pipeline.named_steps['svm']
        weights_df = pd.DataFrame({
            'Feature': X_train.columns,
            'Weight': svm.coef_[0],
            'Absolute_Weight': np.abs(svm.coef_[0]),
            'Participant': participant_id
        })
        weights_data.append(weights_df)
        
        # Evaluate on TEST data
        y_pred = pipeline.predict(X_test)
        test_report = classification_report(y_test, y_pred, output_dict=True)
        
       # Store test metrics - CORRECTED VERSION
        for label, metrics in test_report.items():
            if isinstance(metrics, dict):  # This catches all class-specific metrics
                test_metrics.append({
                    'Participant': participant_id,
                    'Class': str(label),  # Convert label to string in case it's numeric
                    'Precision': metrics['precision'],
                    'Recall': metrics['recall'],
                    'F1': metrics['f1-score'],
                    'Support': metrics['support'],
                    'Accuracy': test_report['accuracy']
                })

# Save all results
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_metrics_df.to_csv(os.path.join(output_dir, f"cv_metrics_{target_variable}_{permutation_label}.csv"), index=False)

test_metrics_df = pd.DataFrame(test_metrics)
test_metrics_df.to_csv(os.path.join(output_dir, f"test_metrics_{target_variable}_{permutation_label}.csv"), index=False)

if weights_data:
    pd.concat(weights_data).to_csv(os.path.join(output_dir, f"feature_weights_{target_variable}_{permutation_label}.csv"), index=False)

print("\nAnalysis complete! All results saved.")


=== Analyzing saved splits ===

Analyzing participant 38
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 71
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 25
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 50
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 31
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 63
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 02
Performing 10-fold CV on train data...
Training on full train data and evaluating on test data...

Analyzing participant 11
Performing 10-fold CV on train data...
Training on full train data and eva

In [ ]:
######################### train-test split without cross-validation ################
# import os
# import pandas as pd
# import numpy as np
# from sklearn.svm import SVC
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, classification_report

# ######################### setting up parameters ################
# ######### need to set up output_dir, permute_y and target_variable

# # Define directories
# input_dir =input_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals v2\post baseline correction"
# output_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results2"


# # Configuration: Set this to True to permute y for permutation testing
# permute_y = False  # Change to False for normal classification

# # Determine report type
# if permute_y:
#     report_type = "permutation_reports"
#     permutation_label = "permutation"
# else:
#     report_type = "classification_reports"
#     permutation_label = "original"
    
# # Store evaluation metrics
# evaluation_metrics = []

# # Ensure output directory exists
# os.makedirs(output_dir, exist_ok=True)

# # Variables to predict
# target_variable = "q1_y"
# excluded_columns = ["q1_y", "q2_y", "q3_y", "q4_y"]
# special_columns = ["Neutral", "Happy", "Sad", "Angry", "Surprised", "Scared", "Disgusted", "Valence", "Arousal"]

# #############################################################
# # Loop through all files in the folder
# for file_name in os.listdir(input_dir):
#     if file_name.endswith(".csv"):
#         file_path = os.path.join(input_dir, file_name)
#         print(f"Processing file: {file_name}")
        
#     # Load data
#     df = pd.read_csv(file_path, dtype={target_variable: str})  # Ensure q3_y is read as string
#     participant_id = file_name.split('.')[0]  # Extract participant number
    
#     print(f"Initial shape of data for participant {participant_id}: {df.shape}")
    
#     # Convert any cell containing the letter 'F' in specified columns to 0
#     for col in special_columns:
#         if col in df.columns:
#             df[col] = df[col].apply(lambda x: 0 if isinstance(x, str) and 'F' in x else (x if isinstance(x, (int, float)) else 0))

#     # Check for non-numeric values in columns except those in excluded_columns
#     non_numeric_columns = {}

#     for col in df.columns:
#         if col not in excluded_columns:
#             non_numeric_values = df[col].apply(lambda x: not isinstance(x, (int, float)) or pd.isna(x)).sum()
#             if non_numeric_values > 0:
#                 non_numeric_columns[col] = non_numeric_values

#     # Convert categorical labels to numeric
#         y = df[target_variable].replace({
#             "long_happy": 4, "long_sad": 3, "fixation": 2,
#             "emotional": 1, "neutral": 0, "Emotional": 1, "Neutral": 0
#         })
        
#     # Print distribution of y after conversion
#     print("Distribution of y after conversion:")
#     print(y.value_counts())
    
#     # Keep only data where y is 1 (emotional) or 0 (neutral)
#     mask = (y == 1) | (y == 0)
#     X = df.drop(columns=excluded_columns)[mask]
#     y = y[mask]
    
#     # Check for non-numeric values in X after filtering
#     non_numeric_columns = {}
#     for col in X.columns:
#             non_numeric_values = X[col].apply(lambda x: not isinstance(x, (int, float)) or pd.isna(x)).sum()
#             if non_numeric_values > 0:
#                 non_numeric_columns[col] = non_numeric_values
    
#     # Check if there is enough data to proceed
#     if y.nunique() < 2:
#         print(f"Warning: Not enough data for classification after filtering. Skipping participant {participant_id}.")
#     else:
#         # Convert to integer
#         y = y.astype(int)
        
#         # Train-test split
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
#         # If permutation is enabled, shuffle the y labels
#         if permute_y:
#             np.random.seed(12)  # Ensure reproducibility
#             y_train = np.random.permutation(y_train.values)
#             y_test = np.random.permutation(y_test.values)
            
#         print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
        
#         # Define ML pipeline
#         pipeline = Pipeline([
#             ('imputer', SimpleImputer(strategy='mean')),
#             ('scaler', StandardScaler()),
#             ('svm', SVC(kernel='linear'))
#         ])
        
#         # Train model
#         pipeline.fit(X_train, y_train)
        
#         # Evaluate model
#         y_pred = pipeline.predict(X_test)
#         accuracy = accuracy_score(y_test, y_pred)
#         report = classification_report(y_test, y_pred)
        
#         print(f"Participant {participant_id} - Accuracy: {accuracy:.4f}")
#         print(f"Classification Report:\n{report}")
        
#         # Save results with target_variable in filenames
#         results_path = os.path.join(output_dir, f"svm_results_{participant_id}_{target_variable}_{permutation_label}.csv")
#         pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).to_csv(results_path, index=False)

#         # Save detailed classification report
#         detailed_reports = {}
#         detailed_reports["SVM"] = report

#         report_dict = classification_report(y_test, y_pred, output_dict=True)
#         for label, metrics in report_dict.items():
#             if isinstance(metrics, dict):
#                 row = {
#                     "Subject": participant_id,
#                     "Model": "SVM",
#                     "Class/Metric": label,
#                     "Precision": metrics.get("precision", None),
#                     "Recall": metrics.get("recall", None),
#                     "F1-Score": metrics.get("f1-score", None),
#                     "Support": metrics.get("support", None),
#                 }
#                 evaluation_metrics.append(row)

#          # Save reports as text file in output_dir with target_variable and permutation status in filename
#         report_file_path = os.path.join(output_dir, f"{participant_id}_{report_type}_{target_variable}.txt")
#         with open(report_file_path, "w") as file:
#             for model_name, report in detailed_reports.items():
#                 file.write(f"Model: {model_name}\n")
#                 file.write(f"{report}\n")
#                 file.write("=" * 40 + "\n")
#             print(f"All reports saved to {report_file_path}")

# # Save all evaluation metrics to CSV in output_dir with target_variable and permutation status in filename
# evaluation_csv_path = os.path.join(output_dir, f"all_participants_evaluation_metrics_{target_variable}_{permutation_label}.csv")
# evaluation_df = pd.DataFrame(evaluation_metrics)
# evaluation_df.to_csv(evaluation_csv_path, index=False)
# print(f"All evaluation metrics saved to {evaluation_csv_path}")

In [ ]:
# ########## Revision one: added output of weights
# import os
# import pandas as pd
# import numpy as np
# from sklearn.svm import SVC
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, classification_report

# ######################### setting up parameters ################
# ######### need to set up output_dir, permute_y and target_variable
# # Define directories
# input_dir =input_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals v2\post baseline correction"
# output_dir = r"\\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\Results\q1_results2"


# # Configuration: Set this to True to permute y for permutation testing
# permute_y = True  # Change to False for normal classification

# # Determine report type
# if permute_y:
#     report_type = "permutation_reports"
#     permutation_label = "permutation"
# else:
#     report_type = "classification_reports"
#     permutation_label = "original"
    
# # Store evaluation metrics
# evaluation_metrics = []
# weights_data = []  # To store weights for all participants

# # Ensure output directory exists
# os.makedirs(output_dir, exist_ok=True)

# # Variables to predict
# target_variable = "q1_y"
# excluded_columns = ["q1_y", "q2_y", "q3_y", "q4_y","stimID"]
# special_columns = ["Neutral", "Happy", "Sad", "Angry", "Surprised", "Scared", "Disgusted", "Valence", "Arousal"]

# #############################################################
# # Loop through all files in the folder
# for file_name in os.listdir(input_dir):
#     if file_name.endswith(".csv"):
#         file_path = os.path.join(input_dir, file_name)
#         print(f"Processing file: {file_name}")
        
#     # Load data
#     df = pd.read_csv(file_path, dtype={target_variable: str})  # Ensure q3_y is read as string
#     participant_id = file_name.split('.')[0]  # Extract participant number
    
#     print(f"Initial shape of data for participant {participant_id}: {df.shape}")
    
#     # Convert any cell containing the letter 'F' in specified columns to 0
#     for col in special_columns:
#         if col in df.columns:
#             df[col] = df[col].apply(lambda x: 0 if isinstance(x, str) and 'F' in x else (x if isinstance(x, (int, float)) else 0))

#     # Check for non-numeric values in columns except those in excluded_columns
#     non_numeric_columns = {}

#     for col in df.columns:
#         if col not in excluded_columns:
#             non_numeric_values = df[col].apply(lambda x: not isinstance(x, (int, float)) or pd.isna(x)).sum()
#             if non_numeric_values > 0:
#                 non_numeric_columns[col] = non_numeric_values

#     # Convert categorical labels to numeric
#         y = df[target_variable].replace({
#             "long_happy": 4, "long_sad": 3, "fixation": 2,
#             "emotional": 1, "neutral": 0, "Emotional": 1, "Neutral": 0
#         })
        
#     # Print distribution of y after conversion
#     print("Distribution of y after conversion:")
#     print(y.value_counts())
    
#     # Keep only data where y is 1 (emotional) or 0 (neutral)
#     mask = (y == 1) | (y == 0)
#     X = df.drop(columns=excluded_columns)[mask]
#     y = y[mask]
    
#     # Check for non-numeric values in X after filtering
#     non_numeric_columns = {}
#     for col in X.columns:
#             non_numeric_values = X[col].apply(lambda x: not isinstance(x, (int, float)) or pd.isna(x)).sum()
#             if non_numeric_values > 0:
#                 non_numeric_columns[col] = non_numeric_values
    
#     # Check if there is enough data to proceed
#     if y.nunique() < 2:
#         print(f"Warning: Not enough data for classification after filtering. Skipping participant {participant_id}.")
#     else:
#         # Convert to integer
#         y = y.astype(int)
        
#         # Train-test split
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
#         # If permutation is enabled, shuffle the y labels
#         if permute_y:
#             np.random.seed(12)  # Ensure reproducibility
#             y_train = np.random.permutation(y_train.values)
#             y_test = np.random.permutation(y_test.values)
            
#         print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
        
#         # Define ML pipeline
#         pipeline = Pipeline([
#             ('imputer', SimpleImputer(strategy='mean')),
#             ('scaler', StandardScaler()),
#             ('svm', SVC(kernel='linear'))
#         ])
        
#         # Train model
#         pipeline.fit(X_train, y_train)
        
#         # Get feature weights from the SVM
#         svm = pipeline.named_steps['svm']
#         feature_weights = svm.coef_[0]
#         feature_names = X_train.columns
        
#         # Create a DataFrame with feature names and weights
#         weights_df = pd.DataFrame({
#             'Feature': feature_names,
#             'Weight': feature_weights,
#             'Absolute_Weight': np.abs(feature_weights)
#         })
        
#         # Sort by absolute weight to find most predictive features
#         sorted_weights = weights_df.sort_values('Absolute_Weight', ascending=False)
        
#         # Print top 5 most predictive variables
#         print("\nTop 5 most predictive variables:")
#         print(sorted_weights.head(5).to_string(index=False))
        
#         # Save weights for this participant
#         weights_df['Participant'] = participant_id
#         weights_data.append(weights_df)
        
#         # Evaluate model
#         y_pred = pipeline.predict(X_test)
#         accuracy = accuracy_score(y_test, y_pred)
#         report = classification_report(y_test, y_pred)
        
#         print(f"Participant {participant_id} - Accuracy: {accuracy:.4f}")
#         print(f"Classification Report:\n{report}")
        
#         # Save results with target_variable in filenames
#         results_path = os.path.join(output_dir, f"svm_results_{participant_id}_{target_variable}_{permutation_label}.csv")
#         pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).to_csv(results_path, index=False)

#         # Save detailed classification report
#         detailed_reports = {}
#         detailed_reports["SVM"] = report

#         report_dict = classification_report(y_test, y_pred, output_dict=True)
#         for label, metrics in report_dict.items():
#             if isinstance(metrics, dict):
#                 row = {
#                     "Subject": participant_id,
#                     "Model": "SVM",
#                     "Class/Metric": label,
#                     "Precision": metrics.get("precision", None),
#                     "Recall": metrics.get("recall", None),
#                     "F1-Score": metrics.get("f1-score", None),
#                     "Support": metrics.get("support", None),
#                 }
#                 evaluation_metrics.append(row)

#          # Save reports as text file in output_dir with target_variable and permutation status in filename
#         report_file_path = os.path.join(output_dir, f"{participant_id}_{report_type}_{target_variable}.txt")
#         with open(report_file_path, "w") as file:
#             for model_name, report in detailed_reports.items():
#                 file.write(f"Model: {model_name}\n")
#                 file.write(f"{report}\n")
#                 file.write("=" * 40 + "\n")
#             print(f"All reports saved to {report_file_path}")

# # Save all evaluation metrics to CSV in output_dir with target_variable and permutation status in filename
# evaluation_csv_path = os.path.join(output_dir, f"all_participants_evaluation_metrics_{target_variable}_{permutation_label}.csv")
# evaluation_df = pd.DataFrame(evaluation_metrics)
# evaluation_df.to_csv(evaluation_csv_path, index=False)
# print(f"All evaluation metrics saved to {evaluation_csv_path}")

# # Save all weights data to CSV
# if weights_data:
#     all_weights_df = pd.concat(weights_data)
#     weights_csv_path = os.path.join(output_dir, f"all_participants_feature_weights_{target_variable}_{permutation_label}.csv")
#     all_weights_df.to_csv(weights_csv_path, index=False)
#     print(f"All feature weights saved to {weights_csv_path}")